# recursive_opt × Trace-Bench — Phase 1→7 Campaign Notebook
One notebook = the whole experiment trajectory: each phase has **(a)** a gate, **(b)** the spec(s),
**(c)** a guarded live run cell, **(d)** analysis (mean±std, paired Δ), **(e)** a **decision**
(ADOPT / REJECT / PARK), and **(f)** **capitalization** — what is recorded into shared campaign
memory and carried forward across runs without relying on hidden defaults.

**Standing decision rule:** ADOPT iff paired same-seed Δ > 1 pooled std on ≥ 2 families at equal
budget; REJECT iff Δ < 0; otherwise PARK.

This notebook is intentionally **live-only**: it fails fast unless `OPENAI_API_KEY` is present,
`gpt-5.4-nano` passes preflight, and a real Trace-Bench adapter is registered. Saved `phase*.json`
files are used only for the final decision board, never as a fallback for failed live cells. Phase
results persist in `./campaign/` and priors/tools/skills in `./mem_campaign`.


In [1]:
import os, sys, json, time, statistics, pathlib

ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "opto").exists() and (ROOT.parent / "opto").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from opto.features.recursive_opt import (run_spec, validate_spec, compile_level,
    agentic_optimizer_factory, MemoryLite, best_config_from, register_config_values)
from opto.features.recursive_opt import tracebench as TB
from opto.features.recursive_opt.inspect_utils import repeat_scores, fmt_mean_std
from opto.features.recursive_opt.runmode import mode_banner, preflight_model

LIVE = bool(os.getenv("OPENAI_API_KEY"))
if not LIVE:
    print("[analysis-only] No OPENAI_API_KEY: run cells validate specs and render saved "
          "results/decisions; no training happens and no numbers are fabricated.")
os.environ["RECURSIVE_OPT_MODEL"] = "gpt-5.4-nano"
os.environ["TRACE_LITELLM_MODEL"] = "gpt-5.4-nano"
os.environ.setdefault("RECURSIVE_OPT_NUM_CANDIDATES", "2")
if LIVE:
    preflight_model("gpt-5.4-nano")

TRACEBENCH = {
    "max_examples": 1,
    "inner_steps": 1,
    "inner_candidates": 1,
    "timeout_seconds": 5,
    "allowed_inner_trainers": ["MinibatchAlgorithm", "PrioritySearch"],
    "eval_kwargs": {"n_train": 1, "n_val": 0},
}
TB.configure_tracebench_adapter(TRACEBENCH, require=True)
if not TB.using_real_tasks():
    raise RuntimeError("A real Trace-Bench adapter is required; synthetic stubs are not allowed.")

RUN_STARTED = time.time()
CAMPAIGN = pathlib.Path("./campaign"); CAMPAIGN.mkdir(exist_ok=True)
MEM_ROOT = "./mem_campaign"
try:
    TB.ensure_eval_only_task_adapter(require=False)
except Exception:
    pass
ADAPTER = TB._TASK_ADAPTER is not None
RUN = LIVE and ADAPTER   # analysis-only without key/adapter: validate + render saved results
print(mode_banner(True))

FAMILIES = {
  "optimization_control": ["llm4ad:online_bin_packing_local", "llm4ad:optimization_admissible_set"],
  "reasoning_control": ["internal:multiobjective_gsm8k", "internal:multi_param"],
}
BUDGET = {
    "optimizer_llm_calls": 12,
    "eval_llm_calls": 24,
    "candidates": 12,
    "wall_time_s": 300,
    "on_exceed": "return_best",
}
SCORING  = {"mode": "relative_delta", "clip": [-1.0, 1.0], "report_raw": True}  # cross-scale safe
PROMOTION= {"enabled": True, "min_support": 2, "min_score": 0.05}  # score-gated: no junk priors
LIVE_SEEDS = (0, 1, 2)  # standing rule needs n>=2; budget.wall_time_s bounds each run
MEASURED_TRAINERS = ["MinibatchAlgorithm", "PrioritySearch"]  # adapter-compatible trainer arms
COMPATIBILITY_ONLY_TRAINERS = ["POLCA", "ParetobasedPS"]  # valid labels, not measured under this smoke allowlist

def save_phase(name, payload): json.dump(payload, open(CAMPAIGN/f"{name}.json","w"), indent=1)
def load_phase(name):
    p = CAMPAIGN/f"{name}.json"
    return json.load(open(p)) if p.exists() else None

def run_variants(make_spec, variants, level_id, seeds=LIVE_SEEDS):
    '''Paired same-seed live runs: one spec per variant; returns {variant: stats}.'''
    out = {}
    for v in variants:
        spec = make_spec(v)
        validate_spec(spec)
        if not RUN:
            print(f"[dry] validated spec for {v}"); continue
        def one(seed, _v=v, _spec=spec):
            s = json.loads(json.dumps(_spec)); s["memory_root"] = MEM_ROOT
            s.setdefault("tracebench", TRACEBENCH)
            s.setdefault("budget", BUDGET)
            return run_spec(s)["results"][level_id(_v)]["score"]
        out[str(v)] = repeat_scores(one, seeds=seeds)
        print(fmt_mean_std(out[str(v)], str(v)))
    return out

def decide(stats, control_key):
    '''ADOPT / REJECT / PARK vs a control variant, per the standing rule.'''
    if not stats or control_key not in stats: return "PARK (no data)"
    c = stats[control_key]; verdicts = {}
    pooled = max(1e-9, statistics.mean([v["std"] for v in stats.values()]))
    for k, v in stats.items():
        if k == control_key: continue
        d = v["mean"] - c["mean"]
        verdicts[k] = "ADOPT" if d > pooled else ("REJECT" if d < 0 else "PARK")
        print(f"  {k:>28}: Δ={d:+.3f} (pooled σ={pooled:.3f}) -> {verdicts[k]}")
    return verdicts

def capitalize(kind, family, content, score, note="", stats=None):
    '''Record a decision/skill/tool into campaign memory so later phases reuse it.
    Pass stats (a repeat_scores dict) to enforce n>=2: single-seed deltas are not evidence.'''
    if stats is not None and stats.get("n", 0) < 2:
        print(f"NOT capitalized [{kind}] {family}: n={stats.get('n')} < 2 (insufficient evidence)")
        return None
    mem = MemoryLite(root=MEM_ROOT)
    rec = mem.record_artifact(level="campaign", family=family, kind=kind,
                              content=str(content), score=float(score),
                              metrics={"note": note} if note else None)
    print(f"capitalized [{kind}] {family}: {str(content)[:60]} (score={score})")
    return rec


[MODE] LIVE LLM run  ·  model = gpt-5.4-nano
  Trace-Bench: REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/notebooks/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])
  Graph/telemetry: AVAILABLE
  Global budget: off: optimizer_llm_calls=0/unlimited, eval_llm_calls=0/unlimited, candidates=0/unlimited, wall_time=0.0s/unlimited, stop_policy=return_best
  Scores below reflect a REAL optimizer run.


## Phase 0 — Gates (no training)
The tests, live model preflight, and real Trace-Bench adapter must be green before any experiment.
**Capitalization:** the gate report itself, so later analysis knows the environment the numbers came
from.

In [2]:

import subprocess, sys
from opto.features.recursive_opt.runmode import trace_io_mode
gates = {}
r = subprocess.run([sys.executable, "-m", "pytest",
    "tests/unit_tests/test_recursive_opt.py", "tests/unit_tests/test_recursive_spec.py", "-q"],
    capture_output=True, text=True, cwd="..") if pathlib.Path("../tests").exists() else     subprocess.run([sys.executable, "-m", "pytest",
    "tests/unit_tests/test_recursive_opt.py", "tests/unit_tests/test_recursive_spec.py", "-q"],
    capture_output=True, text=True)
gates["tests"] = r.stdout.strip().splitlines()[-1] if r.stdout else r.stderr[-200:]
gates["adapter"] = TB.real_mode_status()
gates["trace_io"] = trace_io_mode()
print(json.dumps(gates, indent=1)); save_phase("phase0_gates", gates)


{
 "tests": "70 passed in 2.20s",
 "adapter": "REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/notebooks/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])",
 "trace_io": "AVAILABLE"
}


### Pre-flight: score-spread gate (root-cause guard)
The 0.0-delta stall was a **flat config→score surface** — unplumbed targets (`batch_design`/`memory_policy` never reach the benchmark) plus `inner_steps=0` severing the rest — **not** flat tasks. This gate proves the surface is non-flat on the panel *before* any optimization budget is spent; flat tasks are excluded, not optimized.

In [3]:
from opto.features.recursive_opt import score_spread
PANEL = [t for fam in FAMILIES.values() for t in fam]
spread = {}
if RUN:
    for t in PANEL:
        d = score_spread(t)
        spread[t] = d["spread"]
        print(f"  {t:>40}: spread={d['spread']:.3f}  {'FLAT - excluded' if d['flat'] else 'ok'}")
    PANEL = [t for t in PANEL if spread.get(t, 0) > 0]
    save_phase("phase1_spread", spread)
else:
    print("[dry] spread gate runs with the adapter; previously:", load_phase("phase1_spread"))

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.34it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.33it/s]

[Step 0] Average test score: -2091.8


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

...


/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


[Step 0] Average test score: nan


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00, 5077.85it/s]

[Step 0] Average test score: nan
                      internal:multi_param: spread=0.000  FLAT - excluded



## Phase 1 — Trainer cell *(sets the campaign default under a bounded adapter)*
Specs differ **only** in `fixed.trainer` (trainer is *not* trainable here — paired science),
run under the bounded live smoke budget on `optimization_control`. The measured arms are restricted
to the nested trainers that the current Trace-Bench adapter allowlist can execute cleanly:
`MinibatchAlgorithm` and `PrioritySearch`.

**Lesson learned from probes.** `POLCA` and `ParetobasedPS` are still registered as valid config
labels, but under this smoke budget they are compatibility-only arms: scoring them would measure the
allowlist guard, not trainer performance. Widen `tracebench.allowed_inner_trainers` only when you are
ready for a full, potentially expensive nested benchmark run.

**PR capitalization:** this phase converts a runnable trainer comparison into a reusable campaign
asset: the measured winner becomes `P1_WINNER` (a `kind="decision"` artifact) that later phases read.


In [4]:

TRAINERS = MEASURED_TRAINERS + COMPATIBILITY_ONLY_TRAINERS
register_config_values("trainer", TRAINERS + ["StreamingPrioritySearch"])
def p1_spec(trainer):
    return {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
        "prior_promotion": PROMOTION, "memory_root": MEM_ROOT, "reuse_priors": False,
        "tracebench": TRACEBENCH,
        "levels": [{"id": f"o1_{trainer}", "surface": "config", "family": "optimization_control",
                    # PLUMBED targets only: starting_artifact moves the score even at
                    # inner_steps=0; batch_design/memory_policy search a flat surface.
                    "targets": ["starting_artifact","batch_size"],
                    "constraints": {"starting_artifact": ["", "Answer directly.", "Plan step by step, then answer.", "Plan step by step, then verify the answer before replying."]},
                    "fixed": {"trainer": trainer, "optimizer": "OptoPrimeV2", "trace_type": "internal"},
                    "iterations": 4}]}
print("measured trainer arms:", MEASURED_TRAINERS)
print("compatibility-only under this adapter:", COMPATIBILITY_ONLY_TRAINERS)
p1 = run_variants(p1_spec, MEASURED_TRAINERS, lambda t: f"o1_{t}")
if p1: save_phase("phase1", p1)


measured trainer arms: ['MinibatchAlgorithm', 'PrioritySearch']
compatibility-only under this adapter: ['POLCA', 'ParetobasedPS']
PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  2.14it/s]

...

Evaluating agent:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 1/1 [00:00<00:00,  3.29it/s]

Evaluating agent: 100%|██████████| 1/1 [00:00<00:00,  3.29it/s]

[Step 0] Test/test_score: -2091.8
[Step 0] Algo/Average train score: -2091.8
[Step 0] Update/n_iters: 0
[Step 0] Update/short_term_memory_size: 0
[Step 0] Update/long_term_memory_size: 1
[Step 0] Update/using_short_term_memory: False
[Step 0] Update/using_long_term_memory: True
[Step 0] Update/total_samples: 0
[Step 0] Update/best_candidate_priority: inf
[Step 0] Update/best_candidate_num_rollouts: 0
[Step 0] Update/num_exploration_candidates: 1
[Step 0] Update/exploration_candidates_mean_priority: inf
[Step 0] Update/exploration_candidates_average_num_rollouts: 0.0
[Step 0] Sample/mean_score: -2091.8
[Step 0] Sample/num_samples: 1
[Step 0] Sample/self.n_epochs: 0
[Step 0] Algo/Number of training samples: 1
[Step 0] Parameter/__code:155: import numpy as np

def priority(item: float, bins: np.ndarray) -> np.ndarray:
    """Returns priority with which we want to add item to each bin.
    Args:
        item: Size of item to be added to the bin.
        bins: Array of capacities for each b

PrioritySearch = 0.000 ± 0.000 (n=3)


In [5]:

p1 = load_phase("phase1") or {}
if p1:
    print("Phase 1 — measured trainer comparison (normalized improvement over default cfg)")
    for k,v in p1.items(): print(f"  {k:>22}: {v['mean']:+.3f} ± {v['std']:.3f} (n={v['n']})")
    if COMPATIBILITY_ONLY_TRAINERS:
        print("  compatibility-only, not interpreted as performance:", ", ".join(COMPATIBILITY_ONLY_TRAINERS))
    verdicts = decide(p1, "MinibatchAlgorithm")
    P1_WINNER = max(p1, key=lambda k: p1[k]["mean"])
    capitalize("decision", "*", f"trainer={P1_WINNER}", p1[P1_WINNER]["mean"],
               note="Phase-1 measured winner; campaign default trainer")
else:
    P1_WINNER = "PrioritySearch"; print("no Phase-1 data yet -> default", P1_WINNER)


Phase 1 — measured trainer comparison (normalized improvement over default cfg)
      MinibatchAlgorithm: +0.000 ± 0.000 (n=3)
          PrioritySearch: +0.000 ± 0.000 (n=3)
  compatibility-only, not interpreted as performance: POLCA, ParetobasedPS
                PrioritySearch: Δ=+0.000 (pooled σ=0.000) -> PARK
capitalized [decision] *: trainer=MinibatchAlgorithm (score=0.0)



## Phase 2 — Tracing strategies *(graph/telemetry availability + trace-type discovery)*
The phase validates the `internal`, `otel`, and `hybrid` specs and runs them only
when both conditions are true: the graph/telemetry backend is importable and the
registered benchmark adapter actually plumbs `trace_type` into scoring. Otherwise
it records an explicit skipped status instead of capitalizing a stale assumption.


In [6]:

from opto.features.recursive_opt import traces
from opto.features.recursive_opt.tracebench import current_task_adapter
TRACES = ["internal", "otel", "hybrid"]
def p2_spec(tt):
    s = p1_spec(P1_WINNER); lvl = s["levels"][0]
    lvl["id"] = f"o1_tt_{tt}"; lvl["fixed"]["trace_type"] = tt; return s
for tt in TRACES:
    validate_spec(p2_spec(tt))
adapter = current_task_adapter()
trace_type_plumbed = "trace_type" in getattr(adapter, "PLUMBED_FIELDS", ())
if traces.HAVE_TRACE_IO and trace_type_plumbed:
    p2 = run_variants(p2_spec, TRACES, lambda t: f"o1_tt_{t}")
    if p2:
        save_phase("phase2", p2); decide(p2, "internal")
        disc = {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
                "memory_root": MEM_ROOT, "prior_promotion": PROMOTION, "tracebench": TRACEBENCH,
                "levels": [
                  {"id":"o1_disc","surface":"config","family":"optimization_control",
                   "targets":["trace_type"],"constraints":{"trace_type":TRACES},
                   "fixed":{"trainer":P1_WINNER,"optimizer":"OptoPrimeV2"},"iterations":4},
                  {"id":"o2_mix","surface":"family_policy","family":"*",
                   "targets":["trace_type"],"iterations":2}]}
        validate_spec(disc)
        out = run_spec(disc)
        print("discovered mix:", out["results"]["o2_mix"]["artifact"])
        capitalize("decision","*",out["results"]["o2_mix"]["artifact"],
                   out["results"]["o2_mix"]["score"], note="trace-type discovery; confirm before adopting")
else:
    reason = []
    if not traces.HAVE_TRACE_IO:
        reason.append("graph/telemetry backend unavailable")
    if not trace_type_plumbed:
        reason.append("trace_type is not plumbed by the registered adapter")
    payload = {"status": "skipped", "reason": "; ".join(reason), "validated_specs": TRACES}
    save_phase("phase2", payload)
    print("Phase 2 skipped:", payload["reason"])


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.26it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.25it/s]

...

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.63s/it]

[Step 0] Average test score: 0.0


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00, 7410.43it/s]

[Step 0] Average test score: -1.0
discovered mix: optimization_control => trace_type=internal
reasoning_control => trace_type=internal
capitalized [decision] *: optimization_control => trace_type=internal
reasoning_contro (score=0.004749999999999997)



## Phase 3 — Active search / priors at startup *(transfer)*
Warm vs cold on a **new** family using the priors Phases 1–2 promoted. The promotion **score gate**
(`prior_promotion.min_score`) prevents flat/failed M1 episodes from becoming M3 family priors.

**Lesson learned from probes.** Score-gated promotion is necessary but not sufficient: artifact reuse
can still be neutral or harmful on a flat/new family. This phase now capitalizes memory reuse only when
the paired warm run beats the cold run.


In [7]:

def p3_spec(mode):
    return {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
        "prior_promotion": PROMOTION, "tracebench": TRACEBENCH,
        "memory_root": MEM_ROOT if mode=="warm" else "./mem_cold",
        "reuse_priors": mode=="warm",
        "levels": [{"id": f"o1_{mode}", "surface": "config", "family": "reasoning_control",
                    "targets": ["starting_artifact","batch_size"],
                      "constraints": {"starting_artifact": ["", "Answer directly.", "Plan step by step, then answer.", "Plan step by step, then verify the answer before replying."]},
                    "fixed": {"trainer": P1_WINNER, "optimizer": "OptoPrimeV2"}, "iterations": 4}]}
p3 = run_variants(p3_spec, ["cold","warm"], lambda m: f"o1_{m}")
if p3:
    save_phase("phase3", p3); decide(p3, "cold")
    if "warm" in p3 and p3["warm"]["mean"] > p3["cold"]["mean"]:
        capitalize("decision","reasoning_control",f"reuse_priors Δ={p3['warm']['mean']-p3['cold']['mean']:+.3f}",
                   p3["warm"]["mean"], note="value of memory at startup", stats=p3.get("warm"))
    else:
        print("memory reuse not capitalized: warm did not beat cold under this bounded run")


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]

...

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.36s/it]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.36s/it]

[Step 0] Average test score: 0.0


warm = 0.011 ± 0.017 (n=3)
                          warm: Δ=+0.006 (pooled σ=0.015) -> PARK
capitalized [decision] reasoning_control: reuse_priors Δ=+0.006 (score=0.011333333333333325)


## Phase 4 — AgenticTrace (tool-calling optimizer)
Paired: same spec ± `agentic` (tools wired from memory via `agentic_optimizer_factory`), equal
`optimizer_llm_calls` budget. This **starts the AgenticTrace workstream** with a measurable
baseline. **Capitalization:** tools that helped are saved as `kind="tool"` artifacts — Phase-3
reuse re-arms them automatically in every later run, merged PRs or not.

In [8]:
def p4_spec(mode):
    s = p1_spec(P1_WINNER); lvl = s["levels"][0]
    lvl["id"] = f"o1_{mode}"; s["budget"] = {**BUDGET, "optimizer_llm_calls": 16}
    if mode=="tools": lvl["agentic"] = True; lvl["tools"] = ["trace_search","note"]
    return s
p4 = run_variants(p4_spec, ["plain","tools"], lambda m: f"o1_{m}")
if p4:
    save_phase("phase4", p4); decide(p4, "plain")
    if "tools" in p4 and p4["tools"]["mean"] > p4["plain"]["mean"]:
        capitalize("tool","optimization_control","trace_search", p4["tools"]["mean"],
                   note="tool-evidence improved optimization at equal LLM budget")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.47it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.46it/s]

...

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  2.93it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]

[Step 0] Average test score: -2091.8


tools = 0.000 ± 0.000 (n=3)
                         tools: Δ=+0.000 (pooled σ=0.000) -> PARK



## Phase 5 — Async trainers *(efficiency, not quality)*
Primary metric: **wall-time to a fixed smoke run**; quality must already be tied in Phase 1.
This cell now times only trainer arms supported by the current nested Trace-Bench adapter. Unsupported
or not-yet-integrated trainers are not included, because timing a budget guard/no-op path would produce
misleading near-zero timings.


In [9]:

from opto.features.recursive_opt.optimize import optimize
from opto.features.recursive_opt.tracebench import make_dataset
def p5_run(num_threads, trainer):
    mem = MemoryLite(root=MEM_ROOT)
    lvl = compile_level({"id":"p5","surface":"config","family":"optimization_control",
        "targets":["starting_artifact","batch_size"],
        "fixed":{"trainer":trainer,"optimizer":"OptoPrimeV2"}}, mem, FAMILIES)
    t0 = time.time()
    optimize(lvl, make_dataset([FAMILIES["optimization_control"][0]], repeats=4),
             iterations=4, num_threads=num_threads)
    return time.time()-t0
TIMED_TRAINERS = [P1_WINNER]
if RUN:
    p5 = {f"{tr}/threads={n}": p5_run(n, tr)
          for tr in TIMED_TRAINERS for n in (1, 8)}
    save_phase("phase5", p5)
else:
    p5 = load_phase("phase5") or {}
    print("[dry] timing requires key+adapter; previously:", p5 or "no data")
for k,v in p5.items(): print(f"  {k:>38}: {v:7.1f}s")
if p5: print("Timing is interpreted only for supported trainer paths; quality comes from Phase 1.")


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0
Sampling training minibatch: Sampling 1 agents on 1 inputs (Running sequentially).


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  2.88it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]

[Step 0] Average test score: -2091.8


...

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]

[Step 0] Average test score: -2091.8


Evaluating agent:  25%|██▌       | 1/4 [00:01<00:04,  1.35s/it]

Evaluating agent: 100%|██████████| 4/4 [00:01<00:00,  2.81it/s]

[Step 3] Test/test_score: -2091.8
[Step 3] Algo/Average train score: -2091.8
[Step 3] Update/n_iters: 3
[Step 3] Update/short_term_memory_size: 0
[Step 3] Update/long_term_memory_size: 7
[Step 3] Update/using_short_term_memory: False
[Step 3] Update/using_long_term_memory: True
[Step 3] Update/total_samples: 12
[Step 3] Update/best_candidate_priority: inf
[Step 3] Update/best_candidate_mean_score: -2091.8
[Step 3] Update/best_candidate_num_rollouts: 3
[Step 3] Update/num_exploration_candidates: 1
[Step 3] Update/exploration_candidates_mean_priority: inf
[Step 3] Update/exploration_candidates_mean_score: -2091.8
[Step 3] Update/exploration_candidates_average_num_rollouts: 3.0
[Step 3] Sample/mean_score: -2091.8
[Step 3] Sample/num_samples: 1
[Step 3] Sample/self.n_epochs: 0
[Step 3] Algo/Number of training samples: 4
[Step 3] Parameter/level_config:29: starting_artifact: 
batch_size: 4
            MinibatchAlgorithm/threads=1:    32.5s
            MinibatchAlgorithm/threads=8:    19.9s



## Phase 6 — skills.md (distill → validate as a prior)
Distilled per family from campaign memory (best artifacts + recurring failures) into a structured
SKILL.md; validated with one paired cell (`starting_artifact` "" vs skill).

**Lesson learned from probes.** A distilled skill is an asset only if paired validation improves the
score. Ties are kept as observations but are not capitalized as reusable skills.


In [10]:
def distill_skill(family):
    mem = MemoryLite(root=MEM_ROOT)
    best = mem.best_artifact(family=family)
    fails = mem.similar_failures(family=family, k=3)
    lines = [f"# SKILL - {family}", "", "## Best known setup"]
    if best: lines += [f"(score={best.score:.3f})", "```", str(best.content), "```"]
    lines += ["", "## Known failure modes"] + [f"- {e.feedback[:140]}" for e in fails]
    lines += ["", "## Procedure", "1. Start from the best known setup above.",
              "2. Verify outputs against the failure modes before accepting a candidate."]
    return "\n".join(lines)
SKILL_FAMILY = "optimization_control"
SKILL = distill_skill(SKILL_FAMILY); print(SKILL[:400])
register_config_values("starting_artifact", [SKILL])  # legal arm for validation
def p6_spec(mode):
    s = p1_spec(P1_WINNER); lvl = s["levels"][0]; lvl["id"] = f"o1_{mode}"
    if mode=="skill": lvl["fixed"]["starting_artifact"] = SKILL
    return s
p6 = run_variants(p6_spec, ["plain","skill"], lambda m: f"o1_{m}")
if p6:
    save_phase("phase6", p6); decide(p6, "plain")
    if "skill" in p6 and p6["skill"]["mean"] > p6["plain"]["mean"]:
        capitalize("skill",SKILL_FAMILY,SKILL,p6["skill"]["mean"],
                   note="validated SKILL.md (paired vs empty start)")
    else:
        print("skill not capitalized: paired validation did not improve over plain start")


# SKILL - optimization_control

## Best known setup
(score=0.000)
```
starting_artifact: 
batch_size: 4
```

## Known failure modes
- [real_trace_bench:llm4ad:online_bin_packing_local] inner_steps=1; cfg applied through Trace trainer. trace_type=internal; trace_sources=inte
- [real_trace_bench:llm4ad:online_bin_packing_local] inner_steps=1; cfg applied through Trace trainer. trace_type=internal; t
PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.13it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  3.12it/s]

...

[Step 3] Test/test_score: -1.0
[Step 3] Algo/Average train score: -250000000.75
[Step 3] Update/n_iters: 3
[Step 3] Update/short_term_memory_size: 0
[Step 3] Update/long_term_memory_size: 7
[Step 3] Update/using_short_term_memory: False
[Step 3] Update/using_long_term_memory: True
[Step 3] Update/total_samples: 12
[Step 3] Update/best_candidate_priority: -1.0
[Step 3] Update/best_candidate_mean_score: -1.0
[Step 3] Update/best_candidate_num_rollouts: 3
[Step 3] Update/num_exploration_candidates: 1
[Step 3] Update/exploration_candidates_mean_priority: -1.0
[Step 3] Update/exploration_candidates_mean_score: -1.0
[Step 3] Update/exploration_candidates_average_num_rollouts: 3.0
[Step 3] Sample/mean_score: -1.0
[Step 3] Sample/num_samples: 1
[Step 3] Sample/self.n_epochs: 0
[Step 3] Algo/Number of training samples: 4
[Step 3] Parameter/level_config:35: starting_artifact: Plan step by step, then verify the answer before replying.
batch_size: 4


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00, 61.99it/s]

[Step 0] Average test score: -1000000.0
skill = -1.000 ± 0.000 (n=3)


                         skill: Δ=-1.000 (pooled σ=0.000) -> REJECT
skill not capitalized: paired validation did not improve over plain start


## Phase 7 — Terminal-Bench 2 onboarding *(parallel track — never blocks 1–6)*
TB2 is **not** a Trace-Bench family yet: the accessible path is (a) an adapter honoring the exact
`register_task_adapter` contract on 2–3 sandboxed terminal tasks with deterministic checks, then
(b) treating `terminal` as a new family in the Phase-3 transfer spec, seeded by the Phase-6 skill
and the promoted O3 prior. **Capitalization:** everything Phases 1–6 banked (trainer decision,
trace assumption/mix, priors, tools, skill) is the warm start — TB2 begins where the campaign is,
not from zero.

In [11]:
class TB2AdapterTemplate:
    '''Contract template: implement run_task (and optionally agent_fn) over a local
    terminal harness; scoring must be deterministic checks (cf. make_code_evaluator).'''
    status = "tb2-template (not implemented)"
    def run_task(self, cfg, task_id):
        raise NotImplementedError("wire a sandboxed terminal harness here")
tb2_spec = {"families": {**FAMILIES, "terminal": ["tb2:hello_fs", "tb2:grep_pipeline"]},
    "budget": BUDGET, "scoring": SCORING, "prior_promotion": PROMOTION,
    "memory_root": MEM_ROOT, "reuse_priors": True,
    "levels": [{"id":"o1_tb2","surface":"config","family":"terminal",
                "targets":["starting_artifact","batch_size"],
                "fixed":{"trainer":P1_WINNER,"optimizer":"OptoPrimeV2",
                         "starting_artifact": distill_skill("optimization_control")},
                "iterations": 4}]}
validate_spec(tb2_spec); print("TB2 transfer spec validated — runnable the day the adapter exists.")

TB2 transfer spec validated — runnable the day the adapter exists.


## Campaign decision board

In [12]:

board = {}
for ph in ["phase0_gates","phase1","phase2","phase3","phase4","phase5","phase6"]:
    d = load_phase(ph)
    board[ph] = "no data" if d is None else (d if ph=="phase0_gates" else
        {k:(f"{v['mean']:+.3f}±{v['std']:.3f}" if isinstance(v,dict) and "mean" in v else v)
         for k,v in d.items()})
print(json.dumps(board, indent=1))
mem = MemoryLite(root=MEM_ROOT)
print("\ncapitalized assets (all memory):", mem.summary())
print("current-run capitalized assets:")
for kind in ("decision","assumption","tool","skill"):
    for a in mem.artifact_history(kind=kind):
        if getattr(a, "ts", 0.0) < RUN_STARTED:
            continue
        print(f"  [{kind}] {a.family}: {str(a.content)[:70]} (score={a.score:.3f})")


{
 "phase0_gates": {
  "tests": "70 passed in 2.20s",
  "adapter": "REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/notebooks/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])",
  "trace_io": "AVAILABLE"
 },
 "phase1": {
  "MinibatchAlgorithm": "+0.000\u00b10.000",
  "PrioritySearch": "+0.000\u00b10.000"
 },
 "phase2": {
  "internal": "+0.000\u00b10.000",
  "otel": "+0.000\u00b10.000",
  "hybrid": "+0.000\u00b10.000"
 },
 "phase3": {
  "cold": "+0.005\u00b10.013",
  "warm": "+0.011\u00b10.017"
 },
 "phase4": {
  "plain": "+0.000\u00b10.000",
  "tools": "+0.000\u00b10.000"
 },
 "phase5": {
  "MinibatchAlgorithm/threads=1": 32.45146417617798,
  "MinibatchAlgorithm/threads=8": 19.887879610061646
 },
 "phase6": {
  "plain": "+0.000\u00b10.000",
  "skill": "-1.000\u00b10.000"
 }
}



capitalized assets (all memory): {'episodes': 814, 'artifacts': 51, 'families': ['<multi>', 'llm4ad:online_bin_packing_local', 'optimization_control', 'reasoning_control'], 'priors': {'optimization_control': 1.0, 'reasoning_control': 0.05199999999999999, 'llm4ad:online_bin_packing_local': -2091.8}}
current-run capitalized assets:
  [decision] *: trainer=MinibatchAlgorithm (score=0.000)
  [decision] *: optimization_control => trace_type=internal
reasoning_control => trace (score=0.005)
  [decision] reasoning_control: reuse_priors Δ=+0.006 (score=0.011)
  [tool] optimization_control: trace_search (score=0.000)
  [tool] optimization_control: note (score=0.000)
  [tool] optimization_control: trace_search (score=0.000)
  [tool] optimization_control: note (score=0.000)
  [tool] optimization_control: trace_search (score=0.000)
  [tool] optimization_control: note (score=0.000)
